# Bubble-crop DETR features
Use selected Magi + Paddle text to retain bubbles. Pool frozen DETR encoder tokens for each crop. Preserve original text-query vectors for uncovered text and full-page tail evidence.


In [1]:
from pathlib import Path
import json
import torch
import transformers
import ultralytics
from config import CONFIG, ROOT, BUBBLE_SETTINGS, BASELINE_ROOT, fingerprint
from config import model_files as baseline_model_files
from bubble_features import replace_visuals, sha256, model_files

CACHE = CONFIG['cache_dir']
CACHE.mkdir(parents=True, exist_ok=True)
sequences = json.loads((CACHE / 'sequences.json').read_text(encoding='utf-8'))
base_sequences = json.loads((BASELINE_ROOT / 'cache/sequences.json').read_text(encoding='utf-8'))

def box_identity(sequence):
    pages = [page['sha256'] for page in sequence['pages']]
    boxes = [(box['box_id'], box['bbox']) for box in sequence['boxes']]
    return sequence['sequence_id'], pages, boxes

assert list(map(box_identity, sequences)) == list(map(box_identity, base_sequences))
base_signature = fingerprint({
    'sequences': base_sequences,
    'model': CONFIG['magi_model'],
    'model_files': baseline_model_files(CONFIG['magi_model']),
    'grayscale': CONFIG['grayscale'],
    'text_threshold': CONFIG['text_threshold'],
    'tail_detection_threshold': CONFIG['tail_detection_threshold'],
    'tail_match_threshold': CONFIG['tail_match_threshold'],
    'iou': CONFIG['min_bbox_iou'],
    'version': 3,
})
baseline = torch.load(BASELINE_ROOT / 'cache/magi.pt', map_location='cpu', weights_only=True)
assert baseline['signature'] == base_signature, 'Refresh baseline stage 03 before reusing its fallback features.'

signature = fingerprint({
    'selected_sequences': sequences,
    'baseline': base_signature,
    'settings': BUBBLE_SETTINGS,
    'detector_sha256': sha256(BUBBLE_SETTINGS['bubble_model']),
    'helper_sha256': sha256(ROOT / 'bubble_features.py'),
    'magi_files': model_files(Path(CONFIG['magi_model'])),
    'versions': {'torch': str(torch.__version__), 'transformers': transformers.__version__,
                 'ultralytics': ultralytics.__version__},
})
MAGI_FEATURES = CACHE / 'magi.pt'
saved = torch.load(MAGI_FEATURES, map_location='cpu', weights_only=True) if MAGI_FEATURES.exists() else None
if saved is None or saved.get('signature') != signature:
    vectors, audit = replace_visuals(sequences, BUBBLE_SETTINGS, baseline['vectors'], CACHE / 'bubbles/development')
    saved = dict(signature=signature, vectors=vectors, tails=baseline['tails'],
                 feature_contract='NarrationModelwBubbles', bubble_audit=audit)
    torch.save(saved, MAGI_FEATURES)
print('Visual sequences:', len(saved['vectors']))


RUN seq_2032620aa4e4ac7f page 1: 6/7 bubbles; 6 texts covered; 3 fallback
RUN seq_2032620aa4e4ac7f page 2: 0/2 bubbles; 0 texts covered; 1 fallback
RUN seq_2032620aa4e4ac7f page 3: 7/7 bubbles; 7 texts covered; 1 fallback
RUN seq_952f154fb1505883 page 1: 15/16 bubbles; 15 texts covered; 0 fallback
RUN seq_952f154fb1505883 page 2: 6/6 bubbles; 6 texts covered; 1 fallback
RUN seq_952f154fb1505883 page 3: 15/15 bubbles; 15 texts covered; 0 fallback
RUN seq_7db7a000adeceacf page 1: 10/10 bubbles; 10 texts covered; 0 fallback
RUN seq_7db7a000adeceacf page 2: 10/11 bubbles; 10 texts covered; 1 fallback
RUN seq_7db7a000adeceacf page 3: 10/10 bubbles; 10 texts covered; 1 fallback
RUN seq_9032e5551cf1a9d2 page 1: 6/7 bubbles; 6 texts covered; 1 fallback
RUN seq_9032e5551cf1a9d2 page 2: 11/11 bubbles; 11 texts covered; 2 fallback
RUN seq_9032e5551cf1a9d2 page 3: 13/14 bubbles; 13 texts covered; 4 fallback
RUN seq_2022b8c0e1f2d3c0 page 1: 1/2 bubbles; 1 texts covered; 2 fallback
RUN seq_2022b8c0e